<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/4.2-feature-extraction-rav_and_crema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Feature Extraction: Extendida

En este notebook se realiza el proceso de **extracción de características** y generación de imágenes a partir de dos bases de datos: **RAVDESS** (proceso detallado en el notebook anterior) y **CREMA-D** (*Crowd-sourced Emotional Multimodal Actors Dataset*).

CREMA-D es un conjunto de datos que consta de **7,442 clips originales** interpretados por 91 actores (48 hombres y 43 mujeres) de entre 20 y 74 años, pertenecientes a diversas etnias. Los actores grabaron una selección de 12 oraciones bajo 6 categorías emocionales distintas. Puedes consultar los detalles técnicos aquí: [CREMA-D en Kaggle](https://www.kaggle.com/datasets/ejlok1/cremad).

Como se observará más adelante, debido a que CREMA-D no incluye las emociones *calm* (calma) ni *surprised* (sorpresa), se han tomado **decisiones fundamentales** para el tratamiento de estas clases durante el proceso de extracción:

1. **Fusión de Clases:** Se decidió fusionar la clase *calm* de RAVDESS con la clase *neutral*. Esto se debe a que, en pruebas previas de clasificación (Machine Learning, Deep Learning y pruebas auditivas con personas), la diferencia acústica y perceptual entre ambas es mínima.

2. **Data Augmentation (Surprised):** Dado que la clase *surprised* solo está presente en RAVDESS, se aplicó un proceso exclusivo de **aumento de datos** mediante la inclusión de ruido (*AWGN*) y desplazamiento temporal (*time shift*). Los detalles técnicos se encuentran documentados en las líneas de código correspondientes.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub
import os
# Download latest version desde Kaggle
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
path_crema = kagglehub.dataset_download("ejlok1/cremad")
FAST_ROOT_DIR = '/content/features_local/ravdess-and-crema-images'
BASE_DIR_RAV = "/kaggle/input/ravdess-emotional-speech-audio"
BASE_DIR_CREMA = "/kaggle/input/cremad"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Using Colab cache for faster access to the 'cremad' dataset.


In [12]:
# Import de librerias necesarias
import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
from pandas import DataFrame
import gc
import matplotlib
import os
import logging
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

In [ ]:
# Copiamos la carpeta entera de features desde Drive al disco de Colab
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local
os.makedirs(FAST_ROOT_DIR, exist_ok=True)
# Opcional: Si tienes un archivo .zip en Drive, es AÚN MÁS RÁPIDO copiar el .zip y descomprimirlo localmente:
!cp -r /kaggle/input/ravdess-emotional-speech-audio /content/features_local/ravdess-and-crema-images
!cp -r /kaggle/input/cremad /content/features_local/ravdess-and-crema-images
#!unzip -q /content/ravdess_images_02.zip -d /content/features_local


In [3]:
# Variables de configuración y rutas
#--------------------------------------------------------------------------
SAMPLE_RATE = 22050
MIN_DURATION = 0.5
MAX_DURATION = 3
# En caso de revision de codigo y evitar procesamiento pesado:
GENERATE_CSV = False # Poner en False para desacativar la generacion de archivo CSV
GENERATE_IMAGES = False # Poner en False para desacativar la generacion de Imagenes
PAD_MODE = "constant" # Para padding de audios cortos
# Rutas
#-------------------------------------------------------------------------
OUT_DIR_IMAGES = '/content/drive/MyDrive/ravdess_and_crema_images/' # Drive
OUT_DIR_CSV = '/content/drive/MyDrive/ravdess_features/features_extended.csv'
OUT_DIR_CSV_DARG = '/content/drive/MyDrive/ravdess_features/features_extended_daug.csv'
FAST_ROOT_DIR = '/content/features_local/ravdess-and-crema-images' #Local
OUT_FAST_DIR = '/content/features_local_generadas_csv'
# Parametros
#-------------------------------------------------------------------------
BATCH_SIZE_CSV = 60 # Depende cuanto RAM usemos
IMG_RES = (224,224) # Para ResNet
file_path = FAST_ROOT_DIR
SAMPLE_RATE = 22050
MIN_DURATION = 0.5
MAX_DURATION = 3.0

In [ ]:

os.makedirs(OUT_FAST_DIR, exist_ok=True)
CSV_FILE_PATH = os.path.join(OUT_FAST_DIR, 'features_extended.csv')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', stream=sys.stdout, force=True)

# --- FUNCIONES DE UTILIDAD: Mapeo de emociones en archivos y aplicaion de filtros ---
def get_emotion_from_filename(filename):
    if len(filename.split('-')) == 7:
        emotion_code = int(filename.split('-')[2])
        rav_map = {1: 'neutral', 2: 'neutral', 3: 'happy', 4: 'sad', 5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}
        return rav_map.get(emotion_code, 'unknown')
    elif len(filename.split('_')) >= 3:
        emo_code = filename.split('_')[2].upper()
        crema_map = {'NEU': 'neutral', 'HAP': 'happy', 'SAD': 'sad', 'ANG': 'angry', 'FEA': 'fearful', 'DIS': 'disgust'}
        return crema_map.get(emo_code, 'unknown')
    return 'unknown'

def noise_adder(emotion_data):
    noise_amp = 0.040 * np.random.uniform() * np.amax(emotion_data)
    return emotion_data + noise_amp * np.random.normal(size=emotion_data.shape[0])

def shift(emotion_data):
    shift_range = int(np.random.uniform(low=-5, high=5) * 1000)
    return np.roll(emotion_data, shift_range)

def extract_features(y, sr):
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048)
        mel_spec_db = librosa.power_to_db(mel_spec)
        rmse = librosa.feature.rms(y=y)
        delta = librosa.feature.delta(mfccs)
        delta2 = librosa.feature.delta(mfccs, order=2)

        flat_features = np.concatenate([
            np.mean(mfccs, axis=1), np.std(mfccs, axis=1),
            np.mean(delta, axis=1), np.std(delta, axis=1),
            np.mean(delta2, axis=1), np.std(delta2, axis=1),
            [np.mean(rmse), np.std(rmse), np.mean(mel_spec_db), np.std(mel_spec_db)]
        ])
        return flat_features
    except Exception as e:
        return None

if GENERATE_CSV:

  # --- FUNCIÓN DE TRABAJO PARALELO (Worker) ---
  def process_single_file_for_csv(file_path):
      """
      Procesa un audio y retorna una lista de diccionarios con sus features.
      Retorna lista porque un solo archivo puede generar múltiples filas si hacemos Augmentation.
      """
      filename = os.path.basename(file_path)
      emotion = get_emotion_from_filename(filename)
      if emotion == 'unknown': return []

      try:
          # CORRECCIÓN DE BUG: Carga y preprocesamiento seguros
          y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
          y_trimmed, _ = librosa.effects.trim(y, top_db=45)
          y_norm = librosa.util.normalize(y_trimmed)

          duration = len(y_norm) / sr
          if duration < MIN_DURATION: return [] # Descartado

          target_len = int(MAX_DURATION * sr)
          y_fixed = librosa.util.fix_length(y_norm, size=target_len)

          results = []

          # 1. Feature Original
          feats = extract_features(y_fixed, sr)
          if feats is not None:
              results.append({'features': feats, 'emotion': emotion})

          # 2. Data Augmentation (Solo a 'surprised')
          if emotion == 'surprised':
              # Ruido
              y_noise = noise_adder(y_fixed)
              feats_noise = extract_features(y_noise, sr)
              if feats_noise is not None:
                  results.append({'features': feats_noise, 'emotion': emotion})

              # Shift
              y_shift = shift(y_fixed)
              feats_shift = extract_features(y_shift, sr)
              if feats_shift is not None:
                  results.append({'features': feats_shift, 'emotion': emotion})

          return results
      except Exception as e:
          return []

  # --- MAIN: EJECUCIÓN MULTIPROCESADA ---
  if __name__ == '__main__':
      files_list = [os.path.join(root, f) for root, dirs, files in os.walk(FAST_ROOT_DIR) for f in files if f.endswith('.wav')]
      print(f"Iniciando extracción PARALELA de {len(files_list)} archivos para CSV...")

      all_features = []
      all_labels = []

      with ProcessPoolExecutor() as executor:
          futures = {executor.submit(process_single_file_for_csv, fp): fp for fp in files_list}

          for i, future in enumerate(tqdm(as_completed(futures), total=len(files_list), desc="Generando CSV")):
              result_list = future.result()

              # Agregamos todas las filas generadas por este archivo (1 original + 2 augmentations si es surprised)
              for res in result_list:
                  all_features.append(res['features'])
                  all_labels.append(res['emotion'])

              if i % 1000 == 0:
                  gc.collect()

      print(f"\nProcesamiento finalizado. Se generaron {len(all_features)} filas (incluyendo Data Augmentation).")

      # --- CREACIÓN Y GUARDADO DEL DATAFRAME ---
      columns = ([f'mfcc_mean_{i}' for i in range(20)] + [f'mfcc_std_{i}' for i in range(20)] +
                [f'delta_mean_{i}' for i in range(20)] + [f'delta_std_{i}' for i in range(20)] +
                [f'delta2_mean_{i}' for i in range(20)] + [f'delta2_std_{i}' for i in range(20)] +
                ['rmse_mean', 'rmse_std', 'mel_mean', 'mel_std'])

      df = pd.DataFrame(all_features, columns=columns)
      df['emotion'] = all_labels

      # Escalado (Data Handling)
      scaler = StandardScaler()
      # Separamos características numéricas para no intentar escalar el texto de la columna 'emotion'
      numeric_cols = df.columns[:-1]
      df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

      df.to_csv(CSV_FILE_PATH, index=False)
      print(f"¡Dataframe guardado exitosamente en {CSV_FILE_PATH}!")

else:
  print("La generacion de CSV está desactivada")

La generacion de CSV está desactivada


In [ ]:
# Copiando dataframe a nuestro Drive (opcional)
!zip -r -q /content/features_local_generadas_csv.zip /content/features_local_generadas_csv/
!cp /content/features_local_generadas_csv.zip /content/drive/MyDrive/ravdess_features/features_local_generadas_csv.zip
#

> **Nota:** Se ha aplicado una **eliminación de duplicados** debido a que en la línea **97** —`os.walk(FAST_ROOT_DIR)` de la celda anterior— se realiza un recorrido por el contenido completo de la carpeta de **RAVDESS**. Dado que esta estructura contiene clips duplicados en su interior, se generarían cerca de 1,440 filas con información idéntica. Mantener estos duplicados sería crítico al momento de entrenar el modelo, ya que es una causa directa de **overfitting** (sobreajuste), invalidando la capacidad de generalización del sistema.

In [14]:
df_01 =  pd.read_csv(OUT_DIR_CSV)
df_01 = df_01.drop_duplicates()
df_02 =  pd.read_csv(OUT_DIR_CSV_DARG)
df_02 = df_02 .drop_duplicates()

In [19]:
print ("Contenido del dataframe: \n")
print (f"El tamaño del dataframe sin data augmentation es: {len(df_01)} \n", f"{len(df_01.columns)} Columnas x {len(df_01)} Filas")
print (f"El tamaño del dataframe con data augmentation es: {len(df_02)} \n", f"{len(df_02.columns)} Columnas x {len(df_02)} Filas")

Contenido del dataframe: 

El tamaño del dataframe sin data augmentation es: 8881 
 125 Columnas x 8881 Filas
El tamaño del dataframe con data augmentation es: 9649 
 125 Columnas x 9649 Filas


In [16]:
print("Cantidad de filas por emocion:")

df_01['emotion'].value_counts()

Cantidad de filas por emocion:


,count
emotion,
fearful,1463
angry,1463
disgust,1463
sad,1463
happy,1462
neutral,1375
surprised,192


Vamos a dejar surprised para uso de valanceo de clases

In [20]:
df_02['emotion'].value_counts()

,count
emotion,
angry,1463
fearful,1463
disgust,1463
sad,1463
happy,1462
neutral,1375
surprised,960


In [21]:
#Sobreescribimos los archivos

df_01.to_csv(OUT_DIR_CSV, index=False)
df_02.to_csv(OUT_DIR_CSV_DARG, index=False)

In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
GENERATE_IMAGES = True


TARGET_SAMPLES = int(SAMPLE_RATE * MAX_DURATION) # 66,150 muestras exactas

FEATURES_TO_EXTRACT = ['mel_spec', 'mfcc']
# Para evitar latencia en red que ocurre al guardar directamente en Drive
OUT_DIR_IMAGES_LOCAL = '/content/features_local_generadas'

matplotlib.use('Agg')  # Backend sin GUI: dejar sin comentar para eficiencia

# Crear estructura de carpetas base
for feature in FEATURES_TO_EXTRACT:
    os.makedirs(os.path.join(OUT_DIR_IMAGES_LOCAL, feature), exist_ok=True)


# 1. FUNCIÓN DE GENERACIÓN DE IMÁGENES
def save_images(data, sr, out_path, feature_type, res=IMG_RES):
    # Configuramos la figura sin ejes ni bordes (solo los datos puros para la CNN)
    pixel_width, pixel_height = res[0], res[1]
    my_dpi = 100
    fig = plt.figure(figsize=(pixel_width/my_dpi, pixel_height/my_dpi), dpi=my_dpi)
    # Forzarmos a que el eje ocupe TODA la figura (sin bordes)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    if feature_type == 'mel_spec':
        # Technical Precision: power_to_db es crucial para la percepción logarítmica
        librosa.display.specshow(librosa.power_to_db(data, ref=np.max),
                                 sr=sr, cmap='inferno', ax=ax)
    elif feature_type == 'mfcc':
        librosa.display.specshow(data, sr=sr, cmap='viridis', ax=ax)

    # Guardado optimizado y limpieza estricta de memoria (Troubleshooting RAM)
    plt.savefig(out_path,  transparent=True)
    fig.clear()
    plt.close(fig)

# Encapsular el procesamiento de un solo archivo en una funcion
def process_single_file(file_path):
    ''' Procesa un solo archivo, genera aumento de datos si aplica, y guarda los PNGs. '''
    filename = os.path.basename(file_path)
    emotion = get_emotion_from_filename(filename)

    if emotion == 'unknown':
        return False # Cambiado a False para el contador del Main

    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)

        # TECHNICAL PRECISION: Obligatorio para imágenes, evita distorsión espacial en la CNN
        y_fixed = librosa.util.fix_length(y_trimmed, size=TARGET_SAMPLES)

        # Preparamos un diccionario con las versiones del audio que vamos a procesar
        audio_versions = {'original': y_fixed}

        # Aplicamos Augmentation (Solo a la onda, antes de extraer características)
        if emotion == 'surprised':
            audio_versions['noise'] = librosa.util.fix_length(noise_adder(y_trimmed), size=TARGET_SAMPLES)
            audio_versions['shift'] = librosa.util.fix_length(shift(y_trimmed), size=TARGET_SAMPLES)

        for feature in FEATURES_TO_EXTRACT:
            emotion_dir = os.path.join(OUT_DIR_IMAGES_LOCAL, feature, emotion)
            os.makedirs(emotion_dir, exist_ok=True)

            prefix = "r_" if "-" in filename else "c_"

            # Procesamos cada versión del audio (original, noise, shift)
            for version_name, y_version in audio_versions.items():

                # Asignamos un nombre único si es una versión aumentada
                if version_name == 'original':
                    out_name = f"{prefix}{filename.replace('.wav', '.png')}"
                else:
                    out_name = f"{prefix}{version_name}_{filename.replace('.wav', '.png')}"

                out_path = os.path.join(emotion_dir, out_name)

                if os.path.exists(out_path):
                    continue

                # Data Handling: Extracción de característica 2D
                if feature == 'mel_spec':
                    data = librosa.feature.melspectrogram(y=y_version, sr=sr, n_mels=128, fmax=8000, fmin=20)
                elif feature == 'mfcc':
                    data = librosa.feature.mfcc(y=y_version, sr=sr, n_mfcc=20)

                # Generación y guardado de la imagen final
                save_images(data, sr, out_path, feature)

        return True # El executor as_completed espera un True para contar

    except Exception as e:
        print(f"Error procesando {filename}: {e}")
        return False

if GENERATE_IMAGES:


  # 2. BUCLE PRINCIPAL MULTIPROCESADO
  if __name__ == '__main__':
      files_list = [os.path.join(root, f) for root, dirs, filenames in os.walk(FAST_ROOT_DIR) for f in filenames if f.endswith('.wav')]
      print(f"Iniciando procesamiento PARALELO de {len(files_list)} archivos...")

      processed_count = 0

      # ProcessPoolExecutor crea "clones" de Python para usar todos los núcleos del CPU
      # max_workers=None usa automáticamente todos los núcleos disponibles (usualmente 2 en Colab gratuito)
      with ProcessPoolExecutor() as executor:
          # Mapeamos la función a la lista de archivos y agregamos barra de progreso
          futures = {executor.submit(process_single_file, fp): fp for fp in files_list}

          for future in tqdm(as_completed(futures), total=len(files_list), desc="Generando Imágenes"):
              result = future.result()
              if result is True:
                  processed_count += 1

              # Limpieza periódica obligatoria del proceso principal
              if processed_count % 500 == 0:
                  gc.collect()

      print(f"¡Extracción de imágenes completada! Total procesados con éxito: {processed_count}")

else:
  print("La generacion de imagenes está desactivada")

Iniciando procesamiento PARALELO de 10322 archivos...


Generando Imágenes:   0%|          | 0/10322 [00:00<?, ?it/s]

¡Extracción de imágenes completada! Total procesados con éxito: 10322


In [ ]:
# Guardando el resultado en Drive
'''Para guardar en drive, ya que la informacion en local es volatil
y se pierde cuando de detiene, reinicia o cierra la sesion'''

!zip -r -q /content/features_generadas.zip /content/features_local_generadas

# Mover el archivo zip a Google Drive
!cp /content/features_generadas.zip /content/drive/MyDrive/ravdess_and_crema_images_224_224.zip
print("Copia de seguridad en Drive completada.")


Copia de seguridad en Drive completada.
